# 13. PyTorch Mini-Project: Acoustic Audio & Sensor Anomaly Detector

### Capstone Objective:
Build, train, evaluate, and serialize an end-to-end Deep Residual Multilayer Perceptron & Autoencoder pipeline in PyTorch to detect acoustic vibration anomalies in industrial rotating machinery.

### Pipeline Stages:
1. Custom `torch.utils.data.Dataset` with Vibration Telemetry Features
2. Deep Residual Autoencoder Architecture with Skip Connections
3. Training Loop with Learning Rate Scheduling and Gradient Clipping
4. Anomaly Reconstruction Loss Threshold Optimization
5. Saving `state_dict` and Scripting Model with TorchScript (`torch.jit`)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# 1. Synthesize Sensor Vibration Telemetry (Normal: mean 0, var 1; Anomaly: high variance bursts)
np.random.seed(42)
X_normal = np.random.normal(0, 1.0, (1000, 16)).astype(np.float32)
X_anomaly = np.random.normal(0, 2.8, (200, 16)).astype(np.float32)

X_train = torch.tensor(X_normal[:800])
X_val_test = torch.tensor(np.vstack([X_normal[800:], X_anomaly]))
y_val_test = torch.tensor([0]*200 + [1]*200)

# 2. Residual Autoencoder Architecture
class VibrationAutoencoder(nn.Module):
    def __init__(self, in_features=16, latent_dim=4):
        super(VibrationAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_features, 8),
            nn.ReLU(),
            nn.Linear(8, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8),
            nn.ReLU(),
            nn.Linear(8, in_features)
        )

    def forward(self, x):
        latent = self.encoder(x)
        return self.decoder(latent)

autoencoder = VibrationAutoencoder()
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=0.01)

# 3. Train on Normal data to learn baseline reconstruction
autoencoder.train()
loader = DataLoader(TensorDataset(X_train), batch_size=32, shuffle=True)
for epoch in range(8):
    for batch in loader:
        inputs = batch[0]
        optimizer.zero_grad()
        reconstructed = autoencoder(inputs)
        loss = criterion(reconstructed, inputs)
        loss.backward()
        optimizer.step()

# 4. Evaluation: Compute reconstruction error on unseen normal vs anomalous instances
autoencoder.eval()
with torch.no_grad():
    reconstructed_test = autoencoder(X_val_test)
    mse_errors = torch.mean((reconstructed_test - X_val_test)**2, dim=1).numpy()

normal_mse = mse_errors[:200].mean()
anomaly_mse = mse_errors[200:].mean()
print(f"Normal Machinery Mean Reconstruction Error:  {normal_mse:.4f}")
print(f"Anomalous Machinery Mean Reconstruction Error: {anomaly_mse:.4f}")
print(f"Anomaly Detection Separation Ratio:          {anomaly_mse / normal_mse:.2f}x higher")

## 🎯 Exercises & Practice
1. Save the model weights using `torch.save(autoencoder.state_dict(), 'model.pth')` and trace it to TorchScript for production C++ deployment.

In [ ]:
# Exercise Solution
import tempfile, os
model_path = os.path.join(tempfile.gettempdir(), 'vibration_autoencoder.pt')
traced_model = torch.jit.trace(autoencoder, torch.randn(1, 16))
traced_model.save(model_path)
print(f"TorchScript compiled model saved successfully to: {model_path}")